# Admixture

## Imports


In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from datetime import date,datetime

In [ ]:
d = date.today()
t = datetime.now().time()

print(f'''
DATE: {d} at {t}

pandas=={pd.__version__}
numpy=={np.__version__}
plotly=={plotly.__version__}
''')

## Set directories and variables

#### Common paths

In [ ]:
# Directories

# Main directory
main_dir = "/path/to/home"
MAIN_DIR = main_dir     ### alias

# Data directory
data_dir = f"{main_dir}/data"
DATA_DIR = data_dir     ### alias

# Raw data directory
raw_dir = f"{data_dir}/raw"
RAW_DIR = raw_dir     ### alias

# Meta data (covariate, population, ancestry labels, etc.)
meta_dir = f"{data_dir}/meta"
META_DIR = meta_dir     ### alias

# Projected PCA directory
pca_dir = f"{raw_dir}/PCA"

# Admixture directory
adm_dir = f"{raw_dir}/Admixture"
ADM_DIR = adm_dir     ### alias
os.makedirs(adm_dir, exist_ok=True)

#### Paths to software and tools

In [ ]:
# Hestia NGS Software
tools = "/path/to/tools"

# Plink1.9 and Plink2.0 path
plink = f"{tools}/plink_linux_x86_64_20250615/plink"
plink2 = f"{tools}/plink2_linux_avx2_20250609/plink2"

# GCTA
admixture = f"{tools}/dist/admixture_linux-1.3.0/admixture"

# PCA Projected path
adm_path = f"{tools}/AdmixtureWithReference"
parentals_path = f"{tools}/AdmixtureWithReference/Parentals"

#### Input, output, covariate files

In [ ]:
# Input file (GWASQC unrelated output)
inputPfile = f'{pca_dir}/CATPD_Relationship'

# Covariate file
covar_path = f'{meta_dir}/CATPD_covariate_for_qc.cov'
pop_path = f'{meta_dir}/CATPD_covariate_for_qc.pop'

# GenoTools ancestry predictions
gt_path = f'{meta_dir}/CATPD_qc_ancestry_umap_linearsvc_predicted_labels.txt'

# Pheno Name
pheno = f"DISEASE"

# Output prefix
prefix = f'CATPD_Unrelated'

# Reference panel
# ref_dir = f'{main_dir}/reference/OneThousand/CCDG_14151_B01_GRM_WGS_2020-08-05_chr1_22_X.filtered-phased.vcf.gz'
ref_dir    = f'{main_dir}/reference/GP2_Ref_Panel'
refPfile   = f'{ref_dir}/ref_panel_gp2_prune_rm_underperform_pos_update'
ref_labels = f'{ref_dir}/ref_panel_ancestry_updated.txt'
ref_parentals = f'{ref_dir}/Parentals'

# Threads
threads = 8

In [ ]:
# Empty file for remove samples if there's no outlier list
remove = pd.DataFrame([["NA", "NA"]])
samplesToRemove = f"{meta_dir}/samplestoremove.txt"
remove.to_csv(samplesToRemove, sep="\t", index=False, header=False, na_rep='NA')

### Get PROJECTED PCA

In [ ]:
ref_panel_labels = pd.read_csv(ref_labels, sep='\t', header=None,
                               names=['#FID', 'IID', 'LABEL'])
ref_panel_labels.head()

In [ ]:
for l in ref_panel_labels['LABEL'].unique():
    df = ref_panel_labels[ref_panel_labels['LABEL'] == l]
    df['IID'].to_csv(f'{ref_parentals}/{l.upper()}.txt', sep='\t', header=False, index=False)

In [ ]:
psam_ids = pd.read_csv(f'{inputPfile}.psam', sep='\t', header=0,
                               usecols=['#IID']).rename(columns={'#IID': 'IID'})
psam_ids.insert(0, '#FID', 0)
ref_target = pd.concat([psam_ids, ref_panel_labels])
ref_target['LABEL'] = ref_target['LABEL'].fillna('-')
ref_target.head()

In [ ]:
ref_target.LABEL.value_counts()

In [ ]:
%%time
prepareAdmixture = ["python3", f"{adm_path}/prepareAdmixture.py",
                    "-i", inputPfile,             # Should be plink1.9 bfile 
                    "-I", refPfile,
                    "-o", f'{adm_dir}/{prefix}',  # Should be an empty file with NA\tNA in case nothing to remove, otherwise error
                    "-F", ref_parentals,
                    "-P", plink2,
                    "-p", plink]
d = date.today()
t = datetime.now().time()
print(f"{t} {d}: RUNNING ADMIXTURE")
subprocess.run(prepareAdmixture, check=True)

In [ ]:
runAdmixture    = [ admixture,
                    "--cv", 
                    f'{adm_dir}/{prefix}/LD_Bfile1And2.bed',
                    "10",
                    "-j16",
                    "--supervised"]

d = date.today()
t = datetime.now().time()
print(f"{t} {d}: RUNNING ADMIXTURE")
subprocess.run(runAdmixture, check=True)

## Plots

In [ ]:
Q_FILE = f"{adm_dir}/{prefix}/LD_Bfile1And2.10.Q"       # ADMIXTURE output
FAM_FILE = f"{adm_dir}/{prefix}/LD_Bfile1And2.fam"      # PLINK fam

In [ ]:
# Load fam
fam = pd.read_csv(FAM_FILE, sep=' ', header=None,
                  usecols=[1]).rename(columns={1: 'ID'})

# Merge with popultions
pop = pd.read_csv(pop_path, sep='\t', header=0)
fam = pd.merge(fam, pop, on='ID', how='left')

# Merge with ancestry predictions
gt = pd.read_csv(gt_path, sep='\t', 
                 usecols=['IID', 
                          'label']
                ).rename(columns={'IID': 'ID'})
fam = pd.merge(fam, gt, on='ID', how='inner')

fam.head()

In [ ]:
stats = pd.read_csv(f'{adm_dir}/{prefix}/LD_Bfile1And2.stats', sep=r'\s+', header=None, usecols=[1,6,7,8,9,10,11,12,13,14,15,16])
stats.columns = ['IID', 'label', "AJ", "EUR", "FIN","EAS","AMR","SAS","AFR","AAC","CAS","MDE"]
                 
stats.head()

In [ ]:
labels = ["AAC", "AFR", "AJ","AMR","CAS","EAS","EUR","FIN","MDE","SAS"]

for l in labels:
    df = stats[stats['label'] == l]
    print(l)
    print(df.head(2))
    print('\n\n')

In [ ]:
ANCESTRY_LABELS = {
    0: "AJ",    
    1: "EUR",   
    2: "FIN",
    3: "EAS",
    4: "AMR",
    5: "SAS",
    6: "AFR",
    7: "AAC",
    8: "CAS",
    9: "MDE",
}

ANCESTRY_COLORS = {
     'AAC': '#e377c2',
     'AFR': 'rgb(136, 204, 238)',
     'AJ': 'rgb(204, 102, 119)',
     'AMR': 'rgb(221, 204, 119)',
     'CAS': 'rgb(17, 119, 51)',
     'EAS': 'rgb(51, 34, 136)',
     'EUR': 'rgb(170, 68, 153)',
     'FIN': 'rgb(68, 170, 153)',
     'MDE': 'rgb(153, 153, 51)',
     'SAS': 'rgb(136, 34, 85)',
}

In [ ]:
# Population label comes from the 'POP' column in pop_path
POPULATION_COLUMN = 'POP'

# Populations to display — leave as [] to show ALL populations
SHOW_POPULATIONS = []

# Colour per population (used for the x-axis label tick colour, optional)
color_map = {
    'KAZ': '#636EFA',
    'AZE': '#EF553B',
    'GEO': '#00CC96',
    'ARM': '#AB63FA',
    'TJK': '#FFA15A',
}

PLOT_TITLE      = "Admixture plot"
PLOT_REF_GROUPS = "" 

In [ ]:
def load_admixture_data(q_file, fam_file, pop_path, gt_path, ancestry_labels):
    """
    Load and merge:
      - .Q matrix          → ancestry proportions
      - .fam (col 1 = IID) → sample IDs
      - pop_path           → population labels (ID, POP)
      - gt_path            → predicted ancestry label (IID, label)
    """
    # Q matrix 
    q = pd.read_csv(q_file, sep=r"\s+", header=None)
    q.columns = [ancestry_labels.get(i, f"K{i+1}") for i in range(q.shape[1])]

    # FAM: keep only IID (column index 1) 
    fam = pd.read_csv(fam_file, sep=r"\s+", header=None,
                      usecols=[1]).rename(columns={1: "ID"})

    # Combine fam + Q row-by-row (same order as .fam)
    df = pd.concat([fam.reset_index(drop=True), q.reset_index(drop=True)], axis=1)

    # Population labels 
    pop = pd.read_csv(pop_path, sep="\t", header=0)
    df = pd.merge(df, pop, on="ID", how="left")

    # Predicted ancestry label 
    gt = pd.read_csv(gt_path, sep="\t",
                     usecols=["IID", "label"]).rename(columns={"IID": "ID"})
    df = pd.merge(df, gt, on="ID", how="inner")

    df = df.rename(columns={"ID": "IID"})
    return df


In [ ]:
def sort_within_population(pop_df, sortkey, pop_name):
    """
    Sort descending by one or more ancestry columns.
    sortkey can be:
      - str:  single column, same for all populations
      - dict: {pop_name: [col1, col2, ...]} per-population sort priority
    """
    if isinstance(sortkey, dict):
        cols = sortkey.get(pop_name, [list(pop_df.columns)[0]])
        if isinstance(cols, str):
            cols = [cols]
    else:
        cols = [sortkey]

    return pop_df.sort_values(by=cols, ascending=False)

In [ ]:
def make_admixture_plot(df, ancestry_labels, ancestry_colors,
                        population_column="POP",
                        sortkey=None, show_pops=None,
                        title="Admixture Plot", ref_groups=""):
    """
    Stacked bar admixture plot with one subplot per population, stacked vertically.
 
    Legend swatches are drawn as invisible Scatter traces so they can have a
    thin black outline without affecting the bar borders.
 
    Parameters
    ----------
    df               : DataFrame with IID, POP, ancestry columns, optional label
    ancestry_labels  : dict  {int_index: ancestry_name}
    ancestry_colors  : dict  {ancestry_name: color_string}
    population_column: str   column name holding population codes
    sortkey          : str or dict
                       str  → same single column for all pops, descending
                       dict → {pop_name: [col1, col2, ...]} per-pop priority
                       None → defaults to first ancestry
    show_pops        : list  populations to include; [] / None = all
    title            : str   plot title
    ref_groups       : str   shown in subtitle after '| Ref Groups:'; '' to hide
    """
    ancestries = list(ancestry_labels.values())
 
    if sortkey is None:
        sortkey = ancestries[0]
 
    if show_pops:
        populations = [p for p in show_pops if p in df[population_column].values]
    else:
        populations = df[population_column].unique().tolist()
 
    fig = make_subplots(
        rows=len(populations), cols=1,
        row_heights=[1] * len(populations),
        shared_xaxes=False,
        vertical_spacing=0.06,
    )
 
    # Real bar traces (no border) 
    for row_idx, pop in enumerate(populations, start=1):
        pop_df = df[df[population_column] == pop].copy()
        pop_df = sort_within_population(pop_df, sortkey, pop)
        x_vals = list(range(len(pop_df)))
 
        for ancestry in ancestries:
            hover_label = (
                pop_df["label"].values
                if "label" in pop_df.columns
                else [""] * len(pop_df)
            )
 
            fig.add_trace(
                go.Bar(
                    x=x_vals,
                    y=pop_df[ancestry].values,
                    name=ancestry,
                    marker=dict(
                        color=ancestry_colors.get(ancestry, "#888888"),
                        line=dict(width=0, color="rgba(0,0,0,0)"),  # zero border
                    ),
                    showlegend=False,          # legend handled by scatter below
                    legendgroup=ancestry,
                    hovertemplate=(
                        f"<b>{ancestry}</b><br>"
                        "Sample: %{customdata[0]}<br>"
                        "Predicted: %{customdata[1]}<br>"
                        "Proportion: %{y:.3f}"
                        "<extra></extra>"
                    ),
                    customdata=list(zip(pop_df["IID"].values, hover_label)),
                ),
                row=row_idx, col=1,
            )
 
        fig.update_xaxes(
            title_text=f"<b>{pop}</b>",
            title_font=dict(color="#333333", size=13, family="Arial"),
            showticklabels=False,
            row=row_idx, col=1,
        )
 
    # Fake Scatter traces for legend swatches (outlined squares, no bars) 
    # Placed on row 1 but invisible — only appear in the legend
    for ancestry in ancestries:
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None],
                mode="markers",
                name=ancestry,
                legendgroup=ancestry,
                showlegend=True,
                marker=dict(
                    symbol="square",
                    size=12,
                    color=ancestry_colors.get(ancestry, "#888888"),
                    line=dict(width=1.2, color="#000000"),  # thin black outline
                ),
            ),
            row=1, col=1,
        )
 
    subtitle = f"Groups: {', '.join(populations)}"
    if ref_groups:
        subtitle += f" &nbsp;|&nbsp; Ref Groups: {ref_groups}"
 
    fig.update_layout(
        barmode="stack",
        bargap=0,
        bargroupgap=0,
        font=dict(family="Arial"),
        title=dict(
            text=f"{title}<br><sup>{subtitle}</sup>",
            font=dict(size=16, family="Arial"),
        ),
        legend=dict(
            title=dict(text="Ancestry", font=dict(family="Arial")),
            orientation="v",
            yanchor="middle",
            y=0.5,
            bgcolor="rgba(0,0,0,0)",
            borderwidth=0,
            font=dict(family="Arial"),
        ),
        plot_bgcolor="white",
        height=300 * len(populations),
        width=900,
    )
 
    fig.update_yaxes(range=[0, 1], tickformat=".2f")
 
    mid = (len(populations) // 2) + 1
    fig.update_yaxes(title_text="Percentage Ancestry Estimate", row=mid, col=1)
    bar_traces = [t for t in fig.data if isinstance(t, go.Bar)]
    scatter_traces = sorted(
        [t for t in fig.data if isinstance(t, go.Scatter)],
        key=lambda t: t.name
    )
    fig.data = bar_traces + scatter_traces
    return fig

In [ ]:
df = load_admixture_data(Q_FILE, FAM_FILE, pop_path, gt_path, ANCESTRY_LABELS)
df.head()

In [ ]:
sortkey = {
    "KAZ": ["EUR","FIN",  "CAS", "EAS", "SAS", "MDE"],
    "TJK": ["EAS", "EUR", "FIN",  "SAS"],
    "ARM": ["MDE", "EUR", "AJ"],
    "AZE": ["MDE", "EUR", "AJ"],
    "GEO": ["MDE", "EUR", "AJ"],
}

SHOW_POPULATIONS = ["KAZ", "TJK", "ARM", "AZE", "GEO"]

In [ ]:
fig = make_admixture_plot(
    df,
    ancestry_labels=ANCESTRY_LABELS,
    ancestry_colors=ANCESTRY_COLORS,
    population_column=POPULATION_COLUMN,
    sortkey=sortkey,          # ← pick whichever ancestry makes most sense for your data
    show_pops=SHOW_POPULATIONS if SHOW_POPULATIONS else None,
    title=PLOT_TITLE,
    ref_groups=PLOT_REF_GROUPS,
)
fig.show()